# A02 — P+7 and P+666

Sam Hoyek · CART 498 · Generative Artificial Intelligence

This notebook transforms Wallace Stevens's poem *The Snow Man* with OpenAI GPT-2 (124M parameters). Run the code cells from top to bottom in a fresh Google Colab CPU runtime. Hugging Face downloads the model; no API key is required. Files are written to the current directory.

Each original line is an independent prompt. Its final lexical word is removed, and the remaining prefix has no trailing whitespace. The model predicts the next token. We select the seventh or 666th entry in the descending, unfiltered probability ranking, decode exactly one token, and restore the line's original punctuation. Stanza breaks are preserved. Subwords, unusual spacing, and punctuation tokens are retained rather than repaired. No sampling or random temperature is used.

The last cell checks that both 15-line poems exactly match the approved outputs. The all-noun variation is discussed in reflection.md; it is not implemented here.

Code was prepared with OpenAI Codex. Sam chose rank 666 and supplied the reflection's interpretation. Full attribution and disclosure appear in README.md.


In [1]:
# AI-assisted setup: load GPT-2 for ranked next-token predictions.
%pip -q install transformers==5.17.0

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()
print("GPT-2 loaded. Ready to inspect predictions.")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 loaded. Ready to inspect predictions.


In [2]:
# AI-assisted implementation. Each original line is an independent prompt.
# Replace the last lexical word with exactly one ranked GPT-2 token.
# Keep original punctuation and stanza breaks; do not filter token candidates.
import re
import json
from pathlib import Path

poem = """One must have a mind of winter
To regard the frost and the boughs
Of the pine-trees crusted with snow;

And have been cold a long time
To behold the junipers shagged with ice,
The spruces rough in the distant glitter

Of the January sun; and not to think
Of any misery in the sound of the wind,
In the sound of a few leaves,

Which is the sound of the land
Full of the same wind
That is blowing in the same bare place

For the listener, who listens in the snow,
And, nothing himself, beholds
Nothing that is not there and the nothing that is."""

ranks = [7, 666]
versions = {rank: [] for rank in ranks}
audit = []
for line_number, line in enumerate(poem.splitlines(), start=1):
    if not line:
        for rank in ranks:
            versions[rank].append("")
        continue
    match = re.search(r"([A-Za-z]+)([^A-Za-z]*)$", line)
    assert match, line
    prefix = line[:match.start()].rstrip()
    punctuation = match.group(2)
    encoded = tokenizer(prefix, return_tensors="pt")
    with torch.inference_mode():
        probabilities = model(**encoded).logits[0, -1].softmax(dim=-1)
        values, ids = torch.topk(probabilities, max(ranks))
    for rank in ranks:
        token_id = ids[rank - 1].item()
        token = tokenizer.decode([token_id])
        # GPT-2's decoded token carries its own spacing, including subwords.
        result = prefix + token + punctuation
        versions[rank].append(result)
        audit.append(dict(line=line_number, original=line, prefix=prefix,
                          rank=rank, token_id=token_id, token=token,
                          probability=values[rank - 1].item(), result=result))

assert sum(bool(line) for line in poem.splitlines()) == 15
output_dir = Path(".")
output_dir.mkdir(exist_ok=True)
for rank in ranks:
    text = "\n".join(versions[rank]) + "\n"
    (output_dir / f"P+{rank}.txt").write_text(text, encoding="utf-8")
    print(f"\n{'=' * 16} P+{rank} {'=' * 16}\n{text}")
(output_dir / "token_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
print("Saved P+7.txt, P+666.txt and token_audit.json in the current directory.")



================ P+7 ================
One must have a mind of her
To regard the frost and the death
Of the pine-trees crusted with oil;

And have been cold a long way
To behold the junipers shagged with white,
The spruces rough in the distant horizon

Of the January sun; and not to have
Of any misery in the sound of the sound,
In the sound of a few shots,

Which is the sound of the voice
Full of the same day
That is blowing in the same bare air

For the listener, who listens in the morning,
And, nothing himself, I
Nothing that is not there and the nothing that isn.


================ P+666 ================
One must have a mind of recognition
To regard the frost and the wounds
Of the pine-trees crusted with war;

And have been cold a long med
To behold the junipers shagged with intent,
The spruces rough in the distant marsh

Of the January sun; and not to shy
Of any misery in the sound of the passage,
In the sound of a few British,

Which is the sound of the late
Full of the same fashi

In [3]:
import hashlib
import transformers

expected = {'7': 'One must have a mind of her\nTo regard the frost and the death\nOf the pine-trees crusted with oil;\n\nAnd have been cold a long way\nTo behold the junipers shagged with white,\nThe spruces rough in the distant horizon\n\nOf the January sun; and not to have\nOf any misery in the sound of the sound,\nIn the sound of a few shots,\n\nWhich is the sound of the voice\nFull of the same day\nThat is blowing in the same bare air\n\nFor the listener, who listens in the morning,\nAnd, nothing himself, I\nNothing that is not there and the nothing that isn.\n', '666': 'One must have a mind of recognition\nTo regard the frost and the wounds\nOf the pine-trees crusted with war;\n\nAnd have been cold a long med\nTo behold the junipers shagged with intent,\nThe spruces rough in the distant marsh\n\nOf the January sun; and not to shy\nOf any misery in the sound of the passage,\nIn the sound of a few British,\n\nWhich is the sound of the late\nFull of the same fashion\nThat is blowing in the same bare N\n\nFor the listener, who listens in the commercial,\nAnd, nothing himself, Blake\nNothing that is not there and the nothing that annoy.\n'}
for rank in ranks:
    actual = (output_dir / f"P+{rank}.txt").read_text(encoding="utf-8")
    assert actual == expected[str(rank)], f"P+{rank} differs from the approved output"
    assert len([line for line in actual.splitlines() if line]) == 15
    print(f"P+{rank}: exact match; 15 lines; SHA256 {hashlib.sha256(actual.encode()).hexdigest()}")
print("Versions:", "transformers", transformers.__version__, "torch", torch.__version__)
print("VERIFIED: both required poems reproduce the approved outputs.")


P+7: exact match; 15 lines; SHA256 8e76e74917a498dc096eacfe8ff9d283d12887baae98dcacf3f5df3fe4d7ebb3
P+666: exact match; 15 lines; SHA256 eed2cc8117f4702ba2952ee4384c05325fb42df26b7e88ffd5d2de496d7340c0
Versions: transformers 5.17.0 torch 2.11.0+cpu
VERIFIED: both required poems reproduce the approved outputs.
